# Model Evaluation & Analysis

This notebook provides comprehensive evaluation of the trained AQI prediction model, including:
- Performance metrics (R², MAE, RMSE, MSE)
- Residual analysis
- Prediction accuracy across data ranges
- Feature importance (RandomForest)

**Prerequisites**: Model must be trained via `python ml/train_model.py` or `python ml/tune_and_train.py`

In [ ]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# Setup plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports loaded')

In [ ]:
# Load model and data
MODEL_PATH = 'backend/models/model.joblib'
DATA_PATH = 'ml/sample_data/air_quality_real.csv'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f'Model not found at {MODEL_PATH}. Run: python ml/train_model.py')

model = joblib.load(MODEL_PATH)
df = pd.read_csv(DATA_PATH)

print(f'✓ Model loaded from {MODEL_PATH}')
print(f'✓ Data loaded: {df.shape[0]} samples, {df.shape[1]} columns')
print(f'\nData preview:')
print(df.head())

In [ ]:
# Preprocess data (same as training)
df_clean = df.copy()
df_clean = df_clean.fillna(df_clean.mean(numeric_only=True))
df_clean['temp_humidity_interaction'] = df_clean['temperature'] * df_clean['humidity']
df_clean['temp_rainfall_interaction'] = df_clean['temperature'] * df_clean['rainfall']

feature_cols = [
    'temperature', 'humidity', 'rainfall',
    'temp_humidity_interaction', 'temp_rainfall_interaction'
]
X = df_clean[feature_cols]
y_true = df_clean['aqi']

# Make predictions
y_pred = model.predict(X)

print(f'✓ Preprocessing complete')
print(f'Features: {feature_cols}')

## 1. Performance Metrics

In [ ]:
# Compute metrics
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print('=' * 50)
print('MODEL PERFORMANCE METRICS')
print('=' * 50)
print(f'R² Score:  {r2:.4f}')
print(f'MAE:       {mae:.4f} AQI units')
print(f'RMSE:      {rmse:.4f} AQI units')
print(f'MSE:       {mse:.4f}')
print('=' * 50)

# Interpretation
print(f'\n📊 Interpretation:')
print(f'  - R²: Model explains {r2*100:.1f}% of AQI variance')
print(f'  - MAE: Average prediction error is ±{mae:.1f} AQI units')
print(f'  - RMSE: Root mean squared error is {rmse:.1f} AQI units')

## 2. Residual Analysis

In [ ]:
residuals = y_true - y_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted
axes[0, 0].scatter(y_pred, residuals, alpha=0.5)
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted AQI')
axes[0, 0].set_ylabel('Residual')
axes[0, 0].set_title('Residuals vs Predicted Values')
axes[0, 0].grid(True)

# Residuals histogram
axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residual Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Residuals')
axes[0, 1].grid(True)

# Predicted vs Actual
axes[1, 0].scatter(y_true, y_pred, alpha=0.5)
min_val = min(y_true.min(), y_pred.min())
max_val = max(y_true.max(), y_pred.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[1, 0].set_xlabel('Actual AQI')
axes[1, 0].set_ylabel('Predicted AQI')
axes[1, 0].set_title('Predicted vs Actual')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (Normality Check)')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('ml/plots/comprehensive_residuals.png', dpi=100, bbox_inches='tight')
plt.show()

print('✓ Residuals analysis plot saved to ml/plots/comprehensive_residuals.png')

## 3. Error Distribution by AQI Range

In [ ]:
# Define AQI ranges
ranges = [
    (0, 50, 'Good'),
    (51, 100, 'Moderate'),
    (101, 150, 'Unhealthy for Sensitive Groups'),
    (151, 200, 'Unhealthy'),
    (201, 300, 'Very Unhealthy'),
    (301, 500, 'Hazardous')
]

print('\nError Distribution by AQI Category:')
print('=' * 70)

for low, high, category in ranges:
    mask = (y_true >= low) & (y_true <= high)
    if mask.sum() > 0:
        cat_mae = mean_absolute_error(y_true[mask], y_pred[mask])
        cat_r2 = r2_score(y_true[mask], y_pred[mask])
        print(f'{category:30} | Samples: {mask.sum():4d} | MAE: {cat_mae:6.2f} | R²: {cat_r2:6.3f}')
    else:
        print(f'{category:30} | Samples: {mask.sum():4d} | (no data)')

print('=' * 70)

## 4. Feature Importance (if RandomForest)

In [ ]:
try:
    # Extract RandomForest regressor from pipeline
    if hasattr(model, 'named_steps'):
        rf_reg = model.named_steps.get('regressor', None)
    else:
        rf_reg = model
    
    if hasattr(rf_reg, 'feature_importances_'):
        importances = rf_reg.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = plt.cm.viridis(np.linspace(0, 1, len(feature_cols)))
        bars = ax.barh(range(len(importances)), importances[indices], color=colors)
        ax.set_yticks(range(len(importances)))
        ax.set_yticklabels([feature_cols[i] for i in indices])
        ax.set_xlabel('Importance')
        ax.set_title('Feature Importance (RandomForest)')
        ax.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('ml/plots/feature_importance.png', dpi=100, bbox_inches='tight')
        plt.show()
        
        print('\nFeature Importance:')
        print('=' * 40)
        for i in indices:
            print(f'{feature_cols[i]:30} {importances[i]:.4f}')
        print('=' * 40)
    else:
        print('⚠️  Model does not have feature_importances_ (not a tree-based model)')
except Exception as e:
    print(f'⚠️  Could not extract feature importance: {e}')

## 5. Summary & Recommendations

In [ ]:
print('\n' + '=' * 70)
print('EVALUATION SUMMARY')
print('=' * 70)
print(f'\n📊 Overall Performance:')
print(f'   Model explains {r2*100:.1f}% of AQI variance')
print(f'   Average prediction error: ±{mae:.1f} AQI units')

print(f'\n✓ Residuals:')
print(f'   Mean: {residuals.mean():.4f} (close to 0 is good)')
print(f'   Std:  {residuals.std():.4f}')

# Recommendations
print(f'\n💡 Recommendations:')
if r2 < 0.70:
    print(f'   ⚠️  R² < 0.70: Consider feature engineering or more complex models')
elif r2 < 0.85:
    print(f'   ⚡ R² moderate: Could improve with more features or tuning')
else:
    print(f'   ✓ R² excellent: Model is performing well!')

if mae > 15:
    print(f'   ⚠️  MAE > 15: Predictions may have significant errors for critical decisions')
else:
    print(f'   ✓ MAE acceptable for most use cases')

print(f'\n📈 Next Steps:')
print(f'   1. Validate on unseen data from different regions')
print(f'   2. Test predictions on extreme weather conditions')
print(f'   3. Consider ensemble methods or deep learning for further improvements')
print(f'   4. Deploy with monitoring to track real-world performance')
print('\n' + '=' * 70)